# División del conjunto de datos

En este notebook se muestran algunos de los mecanismos más utilizados para la división del conjunto de datos.

## Conjunto de datos

### Descripción
NSL-KDD es un conjunto de datos propuesto para resolver algunos de los problemas inherentes del conjunto de datos KDD'99. Sigue siendo un benchmark efectivo para comparar métodos de detección de intrusiones.

### Ficheros de datos utilizados
* **KDDTrain+.csv**: Conjunto de entrenamiento completo NSL-KDD en formato CSV

### Referencias
_M. Tavallaee, E. Bagheri, W. Lu, and A. Ghorbani, "A Detailed Analysis of the KDD CUP 99 Data Set," Submitted to Second IEEE Symposium on Computational Intelligence for Security and Defense Applications (CISDA), 2009._

## 1. Lectura del conjunto de datos

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class', 'difficulty_level'
]


def load_kdd_dataset(data_path):
    """Lectura del conjunto de datos NSL-KDD desde CSV."""
    df = pd.read_csv(data_path, header=None, names=COLUMN_NAMES)
    df.drop('difficulty_level', axis=1, inplace=True)
    df['class'] = df['class'].apply(lambda x: 'normal' if x == 'normal' else 'anomaly')
    for col in df.select_dtypes(exclude=['object']).columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

In [ ]:
df = load_kdd_dataset("NSL_KDD-master/KDDTrain+.csv")

In [ ]:
df.info()

## 2. División del conjunto de datos

Se debe separar el conjunto de datos en los diferentes subconjuntos necesarios para realizar los procesos de entrenamiento, validación y pruebas. Sklearn tiene implementada la función **train_test_split**.

In [ ]:
# Separamos el conjunto de datos 60% train set, 40% test set
train_set, test_set = train_test_split(df, test_size=0.4, random_state=42)

In [ ]:
train_set.info()

In [ ]:
test_set.info()

In [ ]:
# Separamos el conjunto de datos de pruebas 50% validation set, 50% test set
val_set, test_set = train_test_split(test_set, test_size=0.5, random_state=42)

In [ ]:
print("Longitud del Training Set:", len(train_set))
print("Longitud del Validation Set:", len(val_set))
print("Longitud del Test Set:", len(test_set))

## 3. Particionado aleatorio y Stratified Sampling

Sklearn tiene implementada la función **train_test_split**, sin embargo, esta función por defecto realiza un particionado del conjunto de datos aleatorio para cada vez que se ejecuta el script. Aún añadiendo una semilla fija para generación aleatoria, cada vez que carguemos de nuevo el conjunto de datos se generarán nuevos subconjuntos. Esto puede ocasionar que después de muchos intentos, el algoritmo "vea" todo el conjunto de datos.

Para solucionar este problema, Sklearn ha introducido el parámetro **shuffle** en la función **train_test_split**.

In [ ]:
# Si shuffle=False, el conjunto de datos no mezclará antes del particionado
train_set, test_set = train_test_split(df, test_size=0.4, random_state=42, shuffle=False)

Estos métodos para dividir el conjunto de datos están bien si tienes un conjunto de datos muy grande, pero si no lo tienes, corres el riesgo de introducir **sampling bias**.

Para evitar esto, se utiliza un método de sampling que se llama **Stratified sampling**. La población es dividida en subconjuntos homogéneos llamados **strata**. El objetivo es que no quede ninguna característica del conjunto de datos sin representación en ninguno de los conjuntos de datos para una o más características en particular.

Sklearn introduce el parámetro **stratify** en la función **train_test_split** para controlar este comportamiento.

> _This stratify parameter makes a split so that the proportion of values in the sample produced will be the same as the proportion of values provided to parameter stratify._
>
> _For example, if variable y is a binary categorical variable with values 0 and 1 and there are 25% of zeros and 75% of ones, stratify=y will make sure that your random split has 25% of 0's and 75% of 1's._

In [ ]:
train_set, test_set = train_test_split(df, test_size=0.4, random_state=42, stratify=df["protocol_type"])

## 4. Generación de una función de particionado

In [ ]:
# Construcción de una función que realice el particionado completo
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

In [ ]:
print("Longitud del conjunto de datos:", len(df))

In [ ]:
train_set, val_set, test_set = train_val_test_split(df, stratify='protocol_type')

In [ ]:
print("Longitud del Training Set:", len(train_set))
print("Longitud del Validation Set:", len(val_set))
print("Longitud del Test Set:", len(test_set))

In [ ]:
# Comprobación de que stratify mantiene la proporción de la característica en los conjuntos
%matplotlib inline
import matplotlib.pyplot as plt
df["protocol_type"].hist()

In [ ]:
train_set["protocol_type"].hist()

In [ ]:
val_set["protocol_type"].hist()

In [ ]:
test_set["protocol_type"].hist()